# ⚡ DevForge — Smart API Integration Tool

> **Hackathon Demo Notebook**

This notebook demonstrates the full DevForge pipeline:

1. 🕷️ **Scrape** — Fetch and clean API documentation from any URL
2. 🧠 **Extract** — Use an LLM to pull out endpoints, auth, and params
3. ⚙️ **Generate** — Produce a ready-to-use Python or JavaScript wrapper class

---
**Team:** DevForge  
**Track:** Gen AI / LLM  
**Problem Statement:** Smart DevTool for API Integration

## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install -q requests beautifulsoup4 lxml google-genai groq openai httpx python-dotenv

## 🔑 Step 2 — Configure API Keys

DevForge uses a **3-tier fallback chain**: Gemini → OpenAI → Groq  
You only need **one key** to run this notebook. Groq is completely free.

Get a free Groq key at [console.groq.com](https://console.groq.com) in 30 seconds.

In [ ]:
import os

# Fill in at least one key
os.environ['GEMINI_API_KEY'] = ''   # aistudio.google.com
os.environ['OPENAI_API_KEY'] = ''   # platform.openai.com
os.environ['GROQ_API_KEY']   = ''   # console.groq.com (FREE)

# Verify at least one is set
keys = {
    'Gemini': os.environ.get('GEMINI_API_KEY'),
    'OpenAI': os.environ.get('OPENAI_API_KEY'),
    'Groq':   os.environ.get('GROQ_API_KEY'),
}
available = [k for k, v in keys.items() if v]
if not available:
    raise ValueError('❌ Please set at least one API key above')
print(f'✅ Available LLM providers: {", ".join(available)}')
print(f'   Priority chain: Gemini → OpenAI → Groq')

## 🕷️ Step 3 — Scraper
Fetches and cleans API documentation from a URL. Strips nav/footer noise, keeps only meaningful content, stays within token budget.

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'}
CONTENT_TAGS = ['p', 'h1', 'h2', 'h3', 'h4', 'li', 'code', 'pre', 'td', 'th']
REMOVE_TAGS  = ['script', 'style', 'nav', 'footer', 'header', 'aside']
MAX_CHARS    = 4000

def fetch_page(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=12)
        r.raise_for_status()
        return r.text
    except Exception as e:
        print(f'  ⚠ Could not fetch {url}: {e}')
        return None

def clean_html(html, char_limit=3000):
    soup = BeautifulSoup(html, 'lxml')
    for tag in soup(REMOVE_TAGS): tag.decompose()
    parts, total, seen = [], 0, set()
    for tag in soup.find_all(CONTENT_TAGS):
        text = tag.get_text(separator=' ', strip=True)
        if not text or len(text) <= 15 or text in seen: continue
        seen.add(text)
        remaining = char_limit - total
        if remaining <= 0: break
        text = text[:remaining]
        parts.append(text)
        total += len(text)
    return '\n'.join(parts)

def discover_links(base_url, html, max_links=1):
    soup = BeautifulSoup(html, 'lxml')
    domain = urlparse(base_url).netloc
    keywords = ['authentication', 'auth', 'reference', 'endpoint', 'api']
    seen, links = set([base_url]), []
    for a in soup.find_all('a', href=True):
        full = urljoin(base_url, a['href'].strip())
        p = urlparse(full)
        if p.netloc != domain or full in seen: continue
        seen.add(full)
        if any(kw in p.path.lower() for kw in keywords):
            links.append(full)
        if len(links) >= max_links: break
    return links

def scrape_docs(url):
    print(f'🕷️  Scraping: {url}')
    html = fetch_page(url)
    if not html: return {'content': '', 'pages': 0}
    main_text = clean_html(html, char_limit=3000)
    all_content = [f'=== MAIN PAGE ===\n{main_text}']
    pages = 1
    for link in discover_links(url, html, max_links=1):
        sub_html = fetch_page(link)
        if not sub_html: continue
        sub_text = clean_html(sub_html, char_limit=1000)
        all_content.append(f'=== {link} ===\n{sub_text}')
        pages += 1
    combined = '\n\n'.join(all_content)[:MAX_CHARS]
    print(f'   ✅ {pages} page(s) scraped — {len(combined)} chars')
    return {'content': combined, 'pages': pages}

print('✅ Scraper ready')

## 🧠 Step 4 — LLM Client (Gemini → OpenAI → Groq fallback)

In [ ]:
import httpx

_gemini = _openai = _groq = None

def get_gemini():
    global _gemini
    if _gemini is None and os.environ.get('GEMINI_API_KEY'):
        from google import genai
        _gemini = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
    return _gemini

def get_openai():
    global _openai
    if _openai is None and os.environ.get('OPENAI_API_KEY'):
        from openai import OpenAI
        _openai = OpenAI(api_key=os.environ['OPENAI_API_KEY'],
                         http_client=httpx.Client(timeout=60.0))
    return _openai

def get_groq():
    global _groq
    if _groq is None and os.environ.get('GROQ_API_KEY'):
        from groq import Groq
        _groq = Groq(api_key=os.environ['GROQ_API_KEY'],
                     http_client=httpx.Client(timeout=60.0))
    return _groq

def llm_generate(prompt, temperature=0.2, max_tokens=4096):
    """Try Gemini → OpenAI → Groq, return (text, provider)."""
    # Gemini
    g = get_gemini()
    if g:
        try:
            from google.genai import types
            r = g.models.generate_content(
                model='gemini-1.5-flash-latest', contents=prompt,
                config=types.GenerateContentConfig(temperature=temperature, max_output_tokens=max_tokens)
            )
            return r.text, 'gemini-1.5-flash'
        except Exception as e:
            print(f'  ⚠ Gemini failed ({type(e).__name__}), trying next...')
    # OpenAI
    o = get_openai()
    if o:
        try:
            r = o.chat.completions.create(model='gpt-4o-mini',
                messages=[{'role':'user','content':prompt}],
                temperature=temperature, max_tokens=max_tokens)
            return r.choices[0].message.content, 'gpt-4o-mini'
        except Exception as e:
            print(f'  ⚠ OpenAI failed ({type(e).__name__}), trying next...')
    # Groq
    gr = get_groq()
    if gr:
        r = gr.chat.completions.create(model='llama-3.3-70b-versatile',
            messages=[{'role':'user','content':prompt}],
            temperature=temperature, max_tokens=max_tokens)
        return r.choices[0].message.content, 'groq/llama-3.3-70b'
    raise RuntimeError('No LLM available — set at least one API key in Step 2')

print('✅ LLM client ready (Gemini → OpenAI → Groq)')

## 🔍 Step 5 — Extractor
Sends scraped docs to the LLM and parses structured API info (endpoints, auth, base URL).

In [ ]:
import json, re

EXTRACT_PROMPT = """Extract API info from this documentation. Reply ONLY with JSON, no extra text.

JSON format:
{{"api_name":"string","base_url":"string","auth":{{"type":"api_key|oauth|bearer|basic|none","header":"string or null","description":"string"}},"endpoints":[{{"method":"GET|POST|PUT|DELETE|PATCH","path":"string","description":"string","params":[{{"name":"string","type":"string","required":true,"description":"string"}}]}}],"raw_summary":"string"}}

Rules: max 8 endpoints, all fields required, guess base_url from domain if missing.
Use case: {use_case}
Docs:
{content}"""

def extract_json(text):
    for attempt in [
        lambda t: json.loads(t.strip()),
        lambda t: json.loads(re.search(r'```(?:json)?\s*([\s\S]+?)```', t).group(1).strip()),
        lambda t: json.loads(t[t.find('{'):t.rfind('}')+1])
    ]:
        try: return attempt(text)
        except: pass
    raise ValueError(f'Could not parse JSON:\n{text[:300]}')

def extract_api_info(content, use_case):
    prompt = EXTRACT_PROMPT.format(use_case=use_case, content=content)
    print(f'🧠 Sending to LLM ({len(prompt)} chars)...')
    text, provider = llm_generate(prompt, temperature=0.1, max_tokens=2048)
    print(f'   ✅ Extracted via {provider}')
    return extract_json(text)

print('✅ Extractor ready')

## ⚙️ Step 6 — Code Generator
Takes the structured API info and generates a production-ready wrapper class.

In [ ]:
PYTHON_PROMPT = """You are a senior Python engineer. Generate a clean Python wrapper class for this API.

API: {api_name} | Base URL: {base_url} | Auth: {auth_type} via {auth_header}
Use case: {use_case}

Endpoints:
{endpoints}

Requirements: use requests library, class named {class_name}Client, constructor takes api_key,
type hints, docstrings, error handling. After the class add: # Example usage
Respond with ONLY Python code, no markdown."""

def make_class_name(api_name):
    return ''.join(w.capitalize() for w in re.sub(r'[^a-zA-Z0-9 ]', ' ', api_name).split() if w)

def format_endpoints(endpoints):
    lines = []
    for ep in endpoints:
        lines.append(f"  {ep['method']} {ep['path']} — {ep['description']}")
    return '\n'.join(lines)

def generate_wrapper(api_info, use_case, language='python'):
    class_name = make_class_name(api_info['api_name'])
    prompt = PYTHON_PROMPT.format(
        api_name=api_info['api_name'],
        base_url=api_info['base_url'],
        auth_type=api_info['auth']['type'],
        auth_header=api_info['auth'].get('header', 'Authorization'),
        use_case=use_case,
        endpoints=format_endpoints(api_info.get('endpoints', [])),
        class_name=class_name,
    )
    print(f'⚙️  Generating {language} wrapper for {api_info["api_name"]}...')
    code, provider = llm_generate(prompt, temperature=0.3, max_tokens=4096)
    code = re.sub(r'^```[a-z]*\n?', '', code.strip())
    code = re.sub(r'\n?```$', '', code)
    print(f'   ✅ Generated via {provider} ({len(code)} chars)')
    return code

print('✅ Generator ready')

## 🚀 Step 7 — Run the Full Pipeline

Change the `API_URL` and `USE_CASE` below and run this cell to generate a wrapper for any API.

In [ ]:
# ── Configure your target API ──────────────────────────────────────
API_URL  = 'https://docs.github.com/en/rest'
USE_CASE = 'Manage repositories and automate CI/CD workflows'
LANGUAGE = 'python'   # 'python' or 'javascript'
# ──────────────────────────────────────────────────────────────────

print('=' * 60)
print('⚡ DevForge Pipeline Starting')
print('=' * 60)

# Step 1: Scrape
scraped = scrape_docs(API_URL)
if not scraped['content']:
    raise RuntimeError('Could not scrape content from URL')

# Step 2: Extract
api_info = extract_api_info(scraped['content'], USE_CASE)

# Step 3: Generate
wrapper_code = generate_wrapper(api_info, USE_CASE, LANGUAGE)

print()
print('=' * 60)
print('✅ Pipeline Complete!')
print('=' * 60)
print(f"API Name : {api_info['api_name']}")
print(f"Base URL : {api_info['base_url']}")
print(f"Auth     : {api_info['auth']['type']} via {api_info['auth'].get('header')}")
print(f"Endpoints: {len(api_info.get('endpoints', []))} found")
print(f"Code     : {len(wrapper_code)} chars generated")

## 📋 Step 8 — View Extracted Endpoints

In [ ]:
print(f"\n📡 {api_info['api_name']} — {len(api_info.get('endpoints',[]))} Endpoints\n")
print(f"Base URL : {api_info['base_url']}")
print(f"Auth     : {api_info['auth']['type'].upper()} → {api_info['auth'].get('header')}")
print(f"Summary  : {api_info['raw_summary']}")
print()
for ep in api_info.get('endpoints', []):
    method = ep['method'].ljust(7)
    print(f"  {method} {ep['path']}")
    print(f"           {ep['description']}")
    for p in ep.get('params', []):
        req = '* ' if p['required'] else '  '
        print(f"           {req}{p['name']} ({p['type']}): {p['description']}")
    print()

## 💻 Step 9 — View Generated Wrapper Code

In [ ]:
print(wrapper_code)

## 💾 Step 10 — Save Generated Code to File

In [ ]:
ext = 'py' if LANGUAGE == 'python' else 'js'
filename = f"{make_class_name(api_info['api_name']).lower()}_client.{ext}"

with open(filename, 'w') as f:
    f.write(wrapper_code)

print(f'✅ Saved to {filename}')
print(f'   Download from the Colab file browser (folder icon on the left)')

---

## 🏗️ Architecture Summary

```
User Input (URL + Use Case)
         ↓
   Scraper (BeautifulSoup)
   • Fetches main page + 1 sub-page
   • Strips noise (nav/footer/scripts)
   • Hard budget: 4,000 chars
         ↓
   LLM Extractor
   • Gemini 1.5 Flash → OpenAI GPT-4o-mini → Groq Llama 3.3
   • Outputs structured JSON: endpoints, auth, base URL
         ↓
   LLM Generator
   • Same fallback chain
   • Produces Python or JavaScript wrapper class
   • Tailored to the user's specific use case
         ↓
   Output: Production-ready wrapper + usage example
```

## 🛠️ Full App
The full DevForge app (React frontend + FastAPI backend + Docker) is available at:  
👉 **https://github.com/INFINIX2004/DevForge**